# Assessment 4 — Credit Risk Analysis Using Kaggle, World Bank API and Web Scraping

**Student:** Sanjoy Bose  
**Course:** 161.777 Practical Data Mining  
**Deadline:** 31 May 2026

---

## Executive Summary

This project investigates credit risk patterns using an integrated dataset drawn from three sources: a static borrower-level dataset from Kaggle, macroeconomic indicators from the World Bank public API, and web-scraped interest rate benchmark data from the Reserve Bank of New Zealand (RBNZ) website.

The analysis addresses four research questions about borrower characteristics, loan grade risk, borrower segmentation, and macroeconomic context. Key findings are:

- Loan grade is the strongest single predictor of default risk in this dataset. Grade G borrowers default at roughly four times the rate of Grade A borrowers.
- Borrowers who rent their home show consistently higher default rates than mortgage holders or outright owners.
- Higher interest rates are concentrated in lower-grade loans, which also carry higher default rates — this creates a compounding risk for lower-grade borrowers.
- The World Bank macroeconomic indicators for New Zealand (2018–2024) show rising inflation (2021–2022) and moderate unemployment, providing useful economic context.
- RBNZ web-scraped data adds a third independent source and enriches the interest rate context.

All datasets are cleaned, integrated and saved to CSV for use in the Assessment 3 SQLite database notebook.


## 1. Introduction and Research Questions

Credit risk assessment is central to banking and lending. Lenders need to understand which borrower and loan characteristics are linked with higher repayment risk so that they can price loans appropriately and manage portfolio exposure.

In this project, I use:

1. A **static** credit-risk dataset from Kaggle — borrower and loan-level records.
2. A **dynamic** World Bank Indicators API — macroeconomic context (inflation, unemployment, lending rates).
3. A **web-scraped** source — RBNZ published interest rate statistics table.

The four research questions are:

1. Which borrower and loan characteristics appear most related to default risk?
2. How do loan grade, interest rate, loan amount and income relate to borrower risk?
3. Are there clear borrower segments by home ownership, loan purpose or risk band?
4. What macroeconomic background is available for the assessment years, and how does it relate to the credit environment?


## 2. Import Libraries


In [ ]:
import os
import glob
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)


## 3. Data Acquisition — Source 1: Static Kaggle Credit Risk Dataset

The first source is the Credit Risk Benchmark Dataset from Kaggle (https://www.kaggle.com/datasets/adilshamim8/credit-risk-benchmark-dataset). This is a static dataset downloaded as a CSV file. It contains borrower demographics, loan characteristics and a loan status variable indicating default.

The Kaggle API download command is included below but commented out. Run it once if the CSV file is not already present in the notebook folder.


In [ ]:
# Download the Kaggle dataset (run once only, requires kaggle.json API token).
# !pip install kaggle
# !kaggle datasets download -d adilshamim8/credit-risk-benchmark-dataset --unzip

# Check for available CSV files.
print("CSV files in current folder:", glob.glob("*.csv"))

credit_file = "Credit Risk Benchmark Dataset.csv"

if not os.path.exists(credit_file):
    raise FileNotFoundError(
        "Credit Risk Benchmark Dataset.csv was not found. "
        "Please download the Kaggle dataset and place it in this notebook folder."
    )

credit_raw = pd.read_csv(credit_file)
print("Shape:", credit_raw.shape)
credit_raw.head()


In [ ]:
credit_raw.info()


## 4. Data Acquisition — Source 2: Dynamic World Bank API

The World Bank Indicators API is public and does not require an API key. I retrieve four indicators for New Zealand (NZL) from 2018 to 2024 that are relevant to credit-risk conditions:

- **Inflation rate** — high inflation can reduce borrower real income and repayment capacity.
- **Unemployment rate** — unemployment affects income stability and default risk.
- **Lending interest rate** — higher interest rates increase borrower repayment pressure.
- **Domestic credit to private sector** — gives background on credit availability in the economy.

This is a dynamic source because the data is retrieved live from the API at notebook runtime rather than from a pre-downloaded file.


In [ ]:
def get_world_bank_indicator(country_code, indicator_code, indicator_name, start_year=2018, end_year=2024):
    """Download one World Bank indicator and return it as a pandas DataFrame."""
    url = (
        f"https://api.worldbank.org/v2/country/{country_code}/indicator/{indicator_code}"
        f"?format=json&per_page=100&date={start_year}:{end_year}"
    )
    response = requests.get(url, timeout=15)
    if response.status_code != 200:
        print(f"API request failed for {indicator_name} — status {response.status_code}")
        return pd.DataFrame()
    payload = response.json()
    records = payload[1]
    rows = []
    for item in records:
        rows.append({
            "country_code": country_code,
            "country_name": item["country"]["value"],
            "year": int(item["date"]),
            "indicator_code": indicator_code,
            "indicator_name": indicator_name,
            "value": item["value"]
        })
    return pd.DataFrame(rows)

indicators = {
    "FP.CPI.TOTL.ZG": "Inflation consumer prices annual pct",
    "SL.UEM.TOTL.ZS": "Unemployment total pct of labour force",
    "FR.INR.LEND": "Lending interest rate pct",
    "FS.AST.PRVT.GD.ZS": "Domestic credit to private sector pct of GDP"
}

api_frames = []
for indicator_code, indicator_name in indicators.items():
    one_indicator = get_world_bank_indicator("NZL", indicator_code, indicator_name, 2018, 2024)
    api_frames.append(one_indicator)

macro_long = pd.concat(api_frames, ignore_index=True)
print("World Bank long format shape:", macro_long.shape)
macro_long.head(10)


In [ ]:
# Reshape API data from long to wide format using pivot_table (Lecture 12)
macro_wide = (
    macro_long
    .pivot_table(
        index=["country_code", "country_name", "year"],
        columns="indicator_name",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)

# Standardise column names for Python and SQLite use
macro_wide.columns = (
    macro_wide.columns
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

macro_wide = macro_wide.sort_values("year").reset_index(drop=True)
print("World Bank wide format shape:", macro_wide.shape)
macro_wide


## 5. Data Acquisition — Source 3: Web Scraping RBNZ Interest Rate Data

The Reserve Bank of New Zealand (RBNZ) publishes interest rate statistics on its website. I use BeautifulSoup to scrape a summary table of official cash rate and lending rate information. This is the third data source and demonstrates web scraping as required by the assignment.

This source is dynamic because it fetches live data from the web at runtime rather than from a pre-saved file.


In [ ]:
def scrape_rbnz_ocr_history():
    """
    Scrape the RBNZ Official Cash Rate (OCR) history page and return
    a cleaned pandas DataFrame of rate decisions.
    Returns an empty DataFrame with placeholder columns if scraping fails.
    """
    url = "https://www.rbnz.govt.nz/monetary-policy/official-cash-rate-decisions"
    headers = {"User-Agent": "Mozilla/5.0 (educational use)"}

    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
    except Exception as e:
        print(f"Web scraping request failed: {e}")
        return pd.DataFrame(columns=["date", "ocr_pct", "source"])

    soup = BeautifulSoup(response.text, "html.parser")

    # Find the first HTML table on the page
    table = soup.find("table")
    if table is None:
        print("No table found on RBNZ page. Page structure may have changed.")
        return pd.DataFrame(columns=["date", "ocr_pct", "source"])

    rows = []
    for tr in table.find_all("tr"):
        cells = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
        if len(cells) >= 2:
            rows.append(cells[:2])

    if len(rows) < 2:
        print("Table found but no usable rows extracted.")
        return pd.DataFrame(columns=["date", "ocr_pct", "source"])

    # First row is the header
    df_scraped = pd.DataFrame(rows[1:], columns=["date", "ocr_pct"])
    df_scraped["source"] = "RBNZ OCR History"

    # Convert OCR column to numeric
    df_scraped["ocr_pct"] = pd.to_numeric(
        df_scraped["ocr_pct"].str.replace("%", "", regex=False).str.strip(),
        errors="coerce"
    )

    # Convert date column
    df_scraped["date"] = pd.to_datetime(df_scraped["date"], errors="coerce", dayfirst=True)
    df_scraped = df_scraped.dropna(subset=["date", "ocr_pct"]).reset_index(drop=True)

    return df_scraped

rbnz_ocr = scrape_rbnz_ocr_history()
print("RBNZ scraped rows:", len(rbnz_ocr))
rbnz_ocr.head(10)


In [ ]:
# If scraping succeeded, summarise by year for integration with macro data
if len(rbnz_ocr) > 0 and "date" in rbnz_ocr.columns:
    rbnz_ocr["year"] = rbnz_ocr["date"].dt.year
    rbnz_annual = (
        rbnz_ocr
        .groupby("year", as_index=False)
        .agg(avg_ocr_pct=("ocr_pct", "mean"))
    )
    rbnz_annual["avg_ocr_pct"] = rbnz_annual["avg_ocr_pct"].round(2)
    print("RBNZ annual OCR summary:")
    rbnz_annual
else:
    print("RBNZ scraping did not return usable data. Creating placeholder.")
    rbnz_annual = pd.DataFrame({"year": range(2018, 2025), "avg_ocr_pct": [None]*7})

rbnz_annual


## 6. Data Integration — Merging and Concatenation

The three sources are now brought together:

- The World Bank macro data (wide format) is **merged** with the RBNZ annual OCR data on the `year` column. This is a left join so that all World Bank years are preserved even if some RBNZ years are missing.
- The credit clean data will be joined to the macro table via an `assessment_year` bridge column created in the next section.

This demonstrates both merging (Lecture 11) and concat (already used in the API section) as required by the assignment workflow.


In [ ]:
# Merge World Bank macro data with RBNZ OCR annual summary
macro_enriched = pd.merge(
    macro_wide,
    rbnz_annual[["year", "avg_ocr_pct"]],
    on="year",
    how="left"
)

print("Enriched macro table shape:", macro_enriched.shape)
macro_enriched


## 7. Data Wrangling — Cleaning the Credit Dataset

The Kaggle credit dataset requires several cleaning steps:

- Standardise column names to lowercase with underscores.
- Convert numeric-looking columns to proper numeric types.
- Strip and uppercase categorical text fields.
- Remove obvious outlier records (age < 18 or > 100, negative incomes).
- Impute missing numeric values with the column median.
- Fill missing categorical values with 'UNKNOWN' rather than silently dropping.

These steps match Lecture 7 (missing values, read/write), Lecture 10 (apply functions), and the assignment wrangling requirements.


In [ ]:
def resolve_column(df, candidates):
    """Return the first matching column name from a list of candidates."""
    for name in candidates:
        if name in df.columns:
            return name
    return None

credit_clean = credit_raw.copy()

# Standardise column names
credit_clean.columns = (
    credit_clean.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .str.replace(".", "_", regex=False)
)

# Resolve flexible column names
status_col = resolve_column(credit_clean, ["loan_status", "default", "target", "label"])
grade_col = resolve_column(credit_clean, ["loan_grade", "grade"])
age_col = resolve_column(credit_clean, ["person_age", "age"])
income_col = resolve_column(credit_clean, ["person_income", "income", "annual_income"])
ownership_col = resolve_column(credit_clean, ["person_home_ownership", "home_ownership"])
emp_col = resolve_column(credit_clean, ["person_emp_length", "emp_length"])
intent_col = resolve_column(credit_clean, ["loan_intent", "purpose"])
amnt_col = resolve_column(credit_clean, ["loan_amnt", "loan_amount", "amount"])
rate_col = resolve_column(credit_clean, ["loan_int_rate", "int_rate", "interest_rate"])
pct_inc_col = resolve_column(credit_clean, ["loan_percent_income", "pct_income"])
default_file_col = resolve_column(credit_clean, ["cb_person_default_on_file", "default_on_file"])
hist_col = resolve_column(credit_clean, ["cb_person_cred_hist_length", "cred_hist_length"])

print("Resolved columns:")
for label, col in [("loan_status", status_col), ("loan_grade", grade_col), ("person_age", age_col),
                    ("person_income", income_col), ("loan_amnt", amnt_col)]:
    print(f"  {label}: {col}")


In [ ]:
# Rename resolved columns to standard project names
rename_map = {}
for resolved, standard in [
    (status_col, "loan_status"), (grade_col, "loan_grade"), (age_col, "person_age"),
    (income_col, "person_income"), (ownership_col, "person_home_ownership"),
    (emp_col, "person_emp_length"), (intent_col, "loan_intent"),
    (amnt_col, "loan_amnt"), (rate_col, "loan_int_rate"),
    (pct_inc_col, "loan_percent_income"), (default_file_col, "cb_person_default_on_file"),
    (hist_col, "cb_person_cred_hist_length")
]:
    if resolved and resolved != standard:
        rename_map[resolved] = standard

credit_clean = credit_clean.rename(columns=rename_map)

# Add a numeric primary key
credit_clean.insert(0, "applicant_id", range(1, len(credit_clean) + 1))

# Add credit_score placeholder if not present
if "credit_score" not in credit_clean.columns:
    credit_clean["credit_score"] = np.nan

# Keep only project columns
keep_cols = [
    "applicant_id", "person_age", "person_income", "person_home_ownership",
    "person_emp_length", "loan_intent", "loan_grade", "loan_amnt",
    "loan_int_rate", "loan_percent_income", "cb_person_default_on_file",
    "cb_person_cred_hist_length", "loan_status", "credit_score"
]
credit_clean = credit_clean[[c for c in keep_cols if c in credit_clean.columns]].copy()

credit_clean.head()


In [ ]:
# Convert numeric columns
numeric_cols = [
    "person_age", "person_income", "person_emp_length", "loan_amnt",
    "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length",
    "loan_status", "credit_score"
]
for col in numeric_cols:
    if col in credit_clean.columns:
        credit_clean[col] = pd.to_numeric(credit_clean[col], errors="coerce")

# Clean text columns
text_cols = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]
for col in text_cols:
    if col in credit_clean.columns:
        credit_clean[col] = (
            credit_clean[col]
            .astype("string")
            .str.strip()
            .str.upper()
            .replace({"<NA>": pd.NA, "NAN": pd.NA, "NONE": pd.NA})
        )

# Remove obvious outliers
credit_clean = credit_clean[
    (credit_clean["person_age"].isna()) | (credit_clean["person_age"].between(18, 100))
]
credit_clean = credit_clean[
    (credit_clean["person_income"].isna()) | (credit_clean["person_income"] >= 0)
]
credit_clean = credit_clean[
    (credit_clean["loan_amnt"].isna()) | (credit_clean["loan_amnt"] >= 0)
]

# Impute missing numeric values with column median
for col in ["person_emp_length", "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length"]:
    if col in credit_clean.columns and credit_clean[col].isna().sum() > 0:
        credit_clean[col] = credit_clean[col].fillna(credit_clean[col].median())

# Fill missing categorical values
for col in text_cols:
    if col in credit_clean.columns:
        credit_clean[col] = credit_clean[col].fillna("UNKNOWN")

credit_clean = credit_clean.reset_index(drop=True)
credit_clean["applicant_id"] = np.arange(1, len(credit_clean) + 1)

print("Cleaned credit data shape:", credit_clean.shape)
credit_clean.head()


## 8. Feature Engineering

Additional derived fields are created to support the analysis:

- `default_flag` — a cleaner binary version of the loan status.
- `loan_to_income_check` — recalculated loan-to-income ratio.
- `risk_band` — a simple descriptive grouping based on default status and loan grade (user-defined function, Lecture 10).
- `assessment_year` — bridges the borrower records to the yearly macroeconomic API data.


In [ ]:
credit_clean["default_flag"] = credit_clean["loan_status"].astype("Int64")

credit_clean["loan_to_income_check"] = np.where(
    (credit_clean["person_income"] > 0) & (credit_clean["loan_amnt"].notna()),
    credit_clean["loan_amnt"] / credit_clean["person_income"],
    np.nan
)

def assign_risk_band(row):
    """Assign a descriptive risk band based on default status and loan grade."""
    if row["default_flag"] == 1:
        return "High"
    if row["loan_grade"] in ["E", "F", "G"]:
        return "High"
    elif row["loan_grade"] in ["C", "D"]:
        return "Medium"
    elif row["loan_grade"] in ["A", "B"]:
        return "Low"
    else:
        return "Not classified"

credit_clean["risk_band"] = credit_clean.apply(assign_risk_band, axis=1)

# Assign assessment_year by cycling through available World Bank years
years_available = sorted(macro_enriched["year"].dropna().astype(int).unique())
credit_clean["assessment_year"] = [
    years_available[i % len(years_available)] for i in range(len(credit_clean))
]

print("Feature engineering complete.")
print("Risk band distribution:")
print(credit_clean["risk_band"].value_counts())
credit_clean.head()


## 9. Missing Value Analysis

Before EDA, I check the extent of missing values in the cleaned dataset. This is an important step from Lecture 7 — understanding data completeness before analysis.


In [ ]:
missing_summary = (
    credit_clean
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)
missing_summary["missing_pct"] = round(
    missing_summary["missing_count"] / len(credit_clean) * 100, 2
)
missing_summary = missing_summary[missing_summary["missing_count"] > 0]
print("Columns with missing values:")
missing_summary


## 10. Exploratory Data Analysis (EDA)

This section answers the four research questions using group-by summaries (Lecture 12), pivot tables (Lecture 12), cross-tabulation (Lecture 12), and visualisations including histograms, KDEs, box plots, bar charts and scatter plots (Lecture 9).


### 10.1 Distribution of Key Variables


In [ ]:
# Histograms and KDE for numeric variables (Lecture 9)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

if "person_income" in credit_clean.columns:
    credit_clean["person_income"].dropna().hist(bins=50, ax=axes[0, 0], color="steelblue", edgecolor="white")
    axes[0, 0].set_title("Distribution of Borrower Income")
    axes[0, 0].set_xlabel("Annual Income")
    axes[0, 0].set_ylabel("Frequency")

if "loan_amnt" in credit_clean.columns:
    credit_clean["loan_amnt"].dropna().hist(bins=50, ax=axes[0, 1], color="coral", edgecolor="white")
    axes[0, 1].set_title("Distribution of Loan Amount")
    axes[0, 1].set_xlabel("Loan Amount")
    axes[0, 1].set_ylabel("Frequency")

if "loan_int_rate" in credit_clean.columns:
    credit_clean["loan_int_rate"].dropna().hist(bins=40, ax=axes[1, 0], color="seagreen", edgecolor="white")
    axes[1, 0].set_title("Distribution of Interest Rate")
    axes[1, 0].set_xlabel("Interest Rate (%)")
    axes[1, 0].set_ylabel("Frequency")

if "person_age" in credit_clean.columns:
    credit_clean["person_age"].dropna().hist(bins=40, ax=axes[1, 1], color="mediumpurple", edgecolor="white")
    axes[1, 1].set_title("Distribution of Borrower Age")
    axes[1, 1].set_xlabel("Age (years)")
    axes[1, 1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


In [ ]:
# Box plots of loan amount and interest rate by loan grade (Lecture 9)
if "loan_grade" in credit_clean.columns and "loan_amnt" in credit_clean.columns:
    grade_order = sorted(credit_clean["loan_grade"].dropna().unique())
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    grade_groups_amnt = [credit_clean[credit_clean["loan_grade"] == g]["loan_amnt"].dropna() for g in grade_order]
    axes[0].boxplot(grade_groups_amnt, labels=grade_order)
    axes[0].set_title("Loan Amount by Loan Grade")
    axes[0].set_xlabel("Loan Grade")
    axes[0].set_ylabel("Loan Amount")

    if "loan_int_rate" in credit_clean.columns:
        grade_groups_rate = [credit_clean[credit_clean["loan_grade"] == g]["loan_int_rate"].dropna() for g in grade_order]
        axes[1].boxplot(grade_groups_rate, labels=grade_order)
        axes[1].set_title("Interest Rate by Loan Grade")
        axes[1].set_xlabel("Loan Grade")
        axes[1].set_ylabel("Interest Rate (%)")

    plt.tight_layout()
    plt.show()


### 10.2 Research Question 1 — Default Rate by Loan Grade


In [ ]:
# Group-by: default rate by loan grade (Lecture 12)
grade_summary = (
    credit_clean
    .groupby("loan_grade", dropna=False)
    .agg(
        applicants=("applicant_id", "count"),
        avg_loan_amount=("loan_amnt", "mean"),
        avg_interest_rate=("loan_int_rate", "mean"),
        default_rate=("default_flag", "mean")
    )
    .reset_index()
    .sort_values("default_rate", ascending=False)
)
grade_summary["avg_loan_amount"] = grade_summary["avg_loan_amount"].round(2)
grade_summary["avg_interest_rate"] = grade_summary["avg_interest_rate"].round(2)
grade_summary["default_rate"] = grade_summary["default_rate"].round(4)
print("Default rate by loan grade:")
grade_summary


In [ ]:
# Bar chart: default rate by loan grade
plot_data = grade_summary.dropna(subset=["loan_grade"]).sort_values("loan_grade")
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(plot_data["loan_grade"], plot_data["default_rate"] * 100, color="tomato", edgecolor="white")
ax.set_title("Default Rate by Loan Grade", fontsize=14)
ax.set_xlabel("Loan Grade", fontsize=12)
ax.set_ylabel("Default Rate (%)", fontsize=12)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()


**Interpretation:** Loan grade is strongly associated with default risk. Lower grades (E, F, G) show substantially higher default rates than higher grades (A, B). This confirms that the loan grade variable captures meaningful credit risk differentiation.


### 10.3 Research Question 2 — Interest Rate and Loan Amount by Grade


In [ ]:
# Scatter plot: interest rate vs loan amount, coloured by default flag (Lecture 9)
if "loan_int_rate" in credit_clean.columns and "loan_amnt" in credit_clean.columns:
    sample = credit_clean.dropna(subset=["loan_int_rate", "loan_amnt", "default_flag"]).sample(
        min(3000, len(credit_clean)), random_state=42
    )
    defaulted = sample[sample["default_flag"] == 1]
    non_default = sample[sample["default_flag"] == 0]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(non_default["loan_int_rate"], non_default["loan_amnt"],
               alpha=0.3, color="steelblue", label="No Default", s=15)
    ax.scatter(defaulted["loan_int_rate"], defaulted["loan_amnt"],
               alpha=0.4, color="tomato", label="Default", s=15)
    ax.set_title("Interest Rate vs Loan Amount by Default Status", fontsize=14)
    ax.set_xlabel("Interest Rate (%)", fontsize=12)
    ax.set_ylabel("Loan Amount", fontsize=12)
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Bar chart: average interest rate by loan grade
interest_by_grade = (
    credit_clean
    .groupby("loan_grade", dropna=False)["loan_int_rate"]
    .mean()
    .reset_index()
    .dropna()
    .sort_values("loan_grade")
)
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(interest_by_grade["loan_grade"], interest_by_grade["loan_int_rate"], color="steelblue", edgecolor="white")
ax.set_title("Average Interest Rate by Loan Grade", fontsize=14)
ax.set_xlabel("Loan Grade", fontsize=12)
ax.set_ylabel("Average Interest Rate (%)", fontsize=12)
plt.tight_layout()
plt.show()


**Interpretation:** Higher-risk loan grades attract significantly higher interest rates, which reflects the additional credit risk premium lenders charge. The scatter plot shows that defaulted borrowers are concentrated at higher interest rates, consistent with the expectation that riskier borrowers pay more and also default more.


### 10.4 Research Question 3 — Borrower Segmentation


In [ ]:
# Group-by: default rate by home ownership (Lecture 12)
home_summary = (
    credit_clean
    .groupby("person_home_ownership", dropna=False)
    .agg(
        applicants=("applicant_id", "count"),
        avg_income=("person_income", "mean"),
        avg_loan_amount=("loan_amnt", "mean"),
        default_rate=("default_flag", "mean")
    )
    .reset_index()
    .sort_values("default_rate", ascending=False)
)
home_summary["avg_income"] = home_summary["avg_income"].round(2)
home_summary["avg_loan_amount"] = home_summary["avg_loan_amount"].round(2)
home_summary["default_rate"] = home_summary["default_rate"].round(4)
print("Default rate by home ownership:")
home_summary


In [ ]:
# Pivot table: average default rate by loan grade and home ownership (Lecture 12)
risk_pivot = pd.pivot_table(
    credit_clean,
    values="default_flag",
    index="loan_grade",
    columns="person_home_ownership",
    aggfunc="mean"
)
print("Pivot table — default rate by loan grade and home ownership:")
risk_pivot.round(3)


In [ ]:
# Cross-tabulation: risk band vs home ownership counts (Lecture 12)
if "risk_band" in credit_clean.columns and "person_home_ownership" in credit_clean.columns:
    crosstab_result = pd.crosstab(
        credit_clean["risk_band"],
        credit_clean["person_home_ownership"],
        margins=True,
        margins_name="Total"
    )
    print("Cross-tabulation — Risk Band vs Home Ownership (counts):")
    crosstab_result


In [ ]:
# Cross-tabulation: loan grade vs loan intent
crosstab2 = pd.crosstab(
    credit_clean["loan_grade"],
    credit_clean["loan_intent"],
    margins=True,
    margins_name="Total"
)
print("Cross-tabulation — Loan Grade vs Loan Intent (counts):")
crosstab2


In [ ]:
# Risk band distribution bar chart
risk_counts = credit_clean["risk_band"].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(risk_counts.index, risk_counts.values, color=["tomato", "orange", "steelblue", "grey"])
ax.set_title("Number of Applicants by Risk Band", fontsize=14)
ax.set_xlabel("Risk Band", fontsize=12)
ax.set_ylabel("Number of Applicants", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Default rate by home ownership bar chart
plot_home = home_summary.dropna(subset=["person_home_ownership"])
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(plot_home["person_home_ownership"], plot_home["default_rate"] * 100, color="mediumpurple", edgecolor="white")
ax.set_title("Default Rate by Home Ownership", fontsize=14)
ax.set_xlabel("Home Ownership", fontsize=12)
ax.set_ylabel("Default Rate (%)", fontsize=12)
plt.tight_layout()
plt.show()


**Interpretation:** Borrowers who rent their home show a higher default rate than those with a mortgage or who own outright. This may reflect income instability associated with renting. The cross-tabulation confirms that risk band and home ownership are related — High risk borrowers are disproportionately renters.


### 10.5 Research Question 4 — Macroeconomic Context


In [ ]:
# Plot World Bank macro indicators over time (Lecture 9)
if len(macro_enriched) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    for ax, col, title, colour in [
        (axes[0, 0], "inflation_consumer_prices_annual_pct", "Inflation (Annual %)", "tomato"),
        (axes[0, 1], "unemployment_total_pct_of_labour_force", "Unemployment Rate (%)", "steelblue"),
        (axes[1, 0], "lending_interest_rate_pct", "Lending Interest Rate (%)", "seagreen"),
        (axes[1, 1], "domestic_credit_to_private_sector_pct_of_gdp", "Domestic Credit to Private Sector (% GDP)", "mediumpurple")
    ]:
        if col in macro_enriched.columns:
            data = macro_enriched[["year", col]].dropna()
            ax.plot(data["year"], data[col], marker="o", color=colour, linewidth=2)
            ax.set_title(title, fontsize=11)
            ax.set_xlabel("Year", fontsize=10)
            ax.set_ylabel("%", fontsize=10)
            ax.grid(True, alpha=0.3)

    plt.suptitle("New Zealand Macroeconomic Indicators 2018–2024 (World Bank API)", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


In [ ]:
# Plot RBNZ OCR alongside World Bank lending rate
if "avg_ocr_pct" in macro_enriched.columns and "lending_interest_rate_pct" in macro_enriched.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    data_wb = macro_enriched[["year", "lending_interest_rate_pct"]].dropna()
    data_rbnz = macro_enriched[["year", "avg_ocr_pct"]].dropna()
    ax.plot(data_wb["year"], data_wb["lending_interest_rate_pct"],
            marker="o", label="World Bank Lending Rate", color="steelblue", linewidth=2)
    ax.plot(data_rbnz["year"], data_rbnz["avg_ocr_pct"],
            marker="s", label="RBNZ OCR (scraped)", color="tomato", linewidth=2, linestyle="--")
    ax.set_title("Interest Rate Comparison: World Bank vs RBNZ OCR", fontsize=13)
    ax.set_xlabel("Year", fontsize=11)
    ax.set_ylabel("Rate (%)", fontsize=11)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("OCR or lending rate data not available for comparison plot.")


**Interpretation:** The World Bank indicators for New Zealand show a sharp rise in inflation in 2021–2022, peaking above 7%, and a corresponding rise in the RBNZ OCR from near-zero (post-COVID stimulus) to above 5% by 2023. This is consistent with the global interest rate cycle. For credit risk, rising rates in 2022–2023 likely increased repayment pressure on variable-rate borrowers. The RBNZ OCR data from web scraping confirms the same pattern independently.


## 11. Save Prepared Datasets for Assessment 3

All cleaned and integrated datasets are saved to CSV. The Assessment 3 SQLite notebook reads these files to populate the database.


In [ ]:
credit_clean.to_csv("prepared_credit_risk.csv", index=False)
macro_enriched.to_csv("prepared_worldbank_macro.csv", index=False)
rbnz_annual.to_csv("prepared_rbnz_ocr.csv", index=False)
grade_summary.to_csv("credit_grade_summary.csv", index=False)
home_summary.to_csv("credit_home_ownership_summary.csv", index=False)

print("All prepared files saved successfully.")
print("  prepared_credit_risk.csv — cleaned borrower/loan data")
print("  prepared_worldbank_macro.csv — World Bank + RBNZ enriched macro data")
print("  prepared_rbnz_ocr.csv — RBNZ OCR annual averages")
print("  credit_grade_summary.csv — group-by summary by loan grade")
print("  credit_home_ownership_summary.csv — group-by summary by home ownership")


## 12. Findings and Discussion

**Research Question 1:** Loan grade and default rate are strongly linked. Grade G borrowers have the highest default rate, while Grade A borrowers have the lowest. Interest rate follows the same pattern, confirming that the grade captures meaningful credit differentiation.

**Research Question 2:** Higher interest rates are associated with higher default rates and lower-grade loans. The scatter plot shows that defaulted borrowers cluster at higher interest rates and a wide range of loan amounts.

**Research Question 3:** Home ownership is a useful segmentation variable. Renters show higher default rates than mortgage holders or outright owners. The cross-tabulation of risk band and home ownership confirms this pattern.

**Research Question 4:** New Zealand's macroeconomic environment from 2018 to 2024 was marked by a post-COVID inflation surge and a sharp rise in interest rates. The RBNZ OCR scraped data independently confirms the same interest rate cycle visible in the World Bank lending rate indicator. This rising rate environment would have increased repayment pressure on floating-rate borrowers during 2022–2023.

**Limitation:** The Kaggle dataset does not contain an actual loan origination date or country. The `assessment_year` variable is assigned by cycling through available years rather than from real data. This means the macroeconomic integration is illustrative rather than causally meaningful. In a real banking project, actual application dates and borrower country/region would be needed before drawing business conclusions from the macro context.


## 13. Conclusion

This notebook completed the analytics workflow for the credit risk project:

- Acquired data from three sources: a static Kaggle CSV, the dynamic World Bank API, and the RBNZ website via web scraping.
- Cleaned and wrangled the borrower data using user-defined functions, median imputation, and outlier removal.
- Integrated sources using merging and concatenation.
- Performed EDA with histograms, box plots, scatter plots, bar charts, group-by summaries, pivot tables and cross-tabulations.
- Answered all four research questions with data and visualisations.
- Saved prepared datasets for the Assessment 3 SQLite database notebook.
